### MLA with Sparse-MoE and shared Experts (DS-v2)  
(Can switch between MoE and Dense mode)  


In [ ]:

import os
from dataclasses import dataclass, field, asdict 
from typing import Optional 
import torch


# -----------------------------------------------------------------------------
# Configuration & Paths
# -----------------------------------------------------------------------------

@dataclass
class ProjectPaths:
    """Centralized configuration for absolute file paths."""
    base_dir: str = os.getcwd() #"./" # Change this to your root production directory

    @property
    def tokenizer_dir(self) -> str:
        return os.path.join(self.base_dir, "custom_tokenizer_2")

    @property
    def model_checkpoints_dir(self) -> str:
        return os.path.join(self.base_dir, "trained_models/mla_v4")

    @property
    def data_corpus(self) -> str:
        return os.path.join(self.base_dir, "dataset/short", "corpus.txt")


@dataclass
class MoEArgs:
    """Configuration specific to the Mixture of Experts layers."""
    num_experts: int = 2
    top_k_experts: int = 1
    z_loss_coef: float = 0.001
    moe_loss_coef: float = 0.01
    hidden_dim_multiplier_perexp: float = 1.0
    first_moe_layer: int = 0 
    num_shared_experts: int = 1
    use_shared_expert: bool = False 


@dataclass
class ModelArgs:
    """Unified configuration for the Transformer architecture."""
    vocab_size: int = 32000  # Fallback example default
    d_model: int = 1024
    num_heads: int = 16
    num_layers: int = 8
    org_context_length: int = 1024
    scale_factor: int = 2 
    yarn_computed_scale: float = 1.0 
    rotary_freq_base: int = 10000
    dropout: float = 0.1
    hidden_dim_multiplier: int = 5 
    moe_args: Optional[MoEArgs] = None
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
    
    # MLA Configuration Maps directly to the model
    q_lora_rank: int = 512 
    kv_lora_rank: int = 256 
    qk_nope_head_dim: int = 128 
    v_head_dim: int = 128 
    qk_rope_head_dim: int = 64

    def __post_init__(self):
        assert self.d_model % self.num_heads == 0, "d_model must be divisible by num_heads" 
        assert self.scale_factor >= 1 
        assert self.q_lora_rank <= self.d_model 
        assert self.kv_lora_rank <= self.d_model 
        assert self.qk_nope_head_dim == self.v_head_dim 

# -----------------------------------------------------------------------------
# Training Configuration
# -----------------------------------------------------------------------------

@dataclass
class TrainingArgs:
    """Strictly training-loop specific parameters."""
    tot_steps:int = 0  # Make it Zero for full run  
    epochs: int = 1 
    batch_size: int = 2
    grad_accum_steps: int = 1
    lr_rate: float = 0.0005
    grad_clip: float = 1.0 
    warmup_steps: int = 100
    save_every_steps: int = 500 


In [ ]:
## utils 

# -----------------------------------------------------------------------------
# Data Processing & Utilities
# -----------------------------------------------------------------------------

from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset
from transformers import AutoTokenizer


## Text Dataset & Memory-Efficient Streaming Utility 
class TextTokenDataset(Dataset):
    """Memory-friendly dataset that reads chunks of text from a corpus."""
    def __init__(self, file_path: str, context_length: int, tokenizer_dir: str):
        self.context_length = context_length
        
        # Real-world fallback logic if no production custom tokenizer exists yet
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
        except Exception:
            print("Custom Tokenizer path missing or invalid. Falling back to gpt2 char/byte simulation...")
            from transformers import GPT2TokenizerFast
            self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        
        if not os.path.exists(file_path):
            raise FileExistsError("Source DATA file not found !") 
            # os.makedirs(os.path.dirname(file_path), exist_ok=True)
            # with open(file_path, "w", encoding="utf-8") as f:
            #     f.write("DeepSeek Multi-Head Latent Attention training script execution example. " * 500)
                
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
            
        self.tokens = self.tokenizer.encode(raw_text)
        self.num_samples = max(0, len(self.tokens) - context_length - 1) 

        tokens = torch.tensor(self.tokens, dtype=torch.long) 
        self.safe_vocab_size = max(len(self.tokenizer), tokens.max().item() + 1)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Grab token sequences offset by 1 for casual autoregressive language modeling targets
        chunk = self.tokens[idx : idx + self.context_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

## Learning Rate Schedule Factory (Cosine with Warmup) 
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(1e-5, 0.5 * (1.0 + math.cos(math.pi * progress))) 
    return LambdaLR(optimizer, lr_lambda)


def calculate_active_params_MoE(model):
    """Calculates total, active and shared parameters for a model using hybrid sparse MoE layers 
    """ 
    total_params = 0
    routed_exp_param = 0
    shared_exp_param = 0 

    num_experts = None 
    num_routed_experts = None 
    top_k = None 
    shared_expert = False 

    # 1. Safely calculate total parameters
    for param in model.parameters():
        if param.requires_grad:
            total_params += param.numel()

    # 2. Iterate strictly through modules to prevent string-matching collisions with dense layers
    for name, module in model.named_modules():
        # --- Standard MoE Check ---
        if isinstance(module, MoEFeedForward):
            if num_experts is None:
                num_experts = module.num_experts 
                top_k = module.num_experts_per_tok 
            
            # These are batched nn.Parameters, so we check them directly
            for p_name, param in module.named_parameters(recurse=False):
                if p_name in ["w1", "w2", "w3"] and param.requires_grad:
                    routed_exp_param += param.numel()
        # --- DeepSeek Sparse MoE Check ---
        elif isinstance(module, DeepSeekSparseMoE):
            if num_routed_experts is None:
                num_routed_experts = module.n_routed_experts 
                top_k = module.num_experts_per_tok  
                shared_expert = True 
            
            # 1. Count Routed Experts (batched nn.Parameters)
            for p_name, param in module.named_parameters(recurse=False):
                if p_name in ["w1", "w2", "w3"] and param.requires_grad:
                    routed_exp_param += param.numel()
                    
            # 2. Count Shared Experts (nn.Linear layers)
            for shared_module in [module.shared_w1, module.shared_w2, module.shared_w3]:
                for param in shared_module.parameters():
                    if param.requires_grad:
                        shared_exp_param += param.numel()

    # Safety check if no MoE layer was found
    if (num_experts is None) and (num_routed_experts is None):
        raise ValueError("Could not find any MoEFeedForward or DeepSeekSparseMoE layers in the model.")

    total_expert_params = routed_exp_param + shared_exp_param 
    base_params = total_params - total_expert_params 

    # 3. Math to find active parameters
    if shared_expert:
        # Divide routed expert parameters by the number of experts to find 1 routed expert's size
        # (This scales perfectly even if you have multiple MoE layers)
        single_routedexp_size = routed_exp_param // num_routed_experts 
        
        # DeepSeek Rule: ALL shared experts are active for EVERY token. 
        active_shared = shared_exp_param
        active_routed = single_routedexp_size * top_k

        active_params = base_params + active_shared + active_routed

        return {
            "total_parameters": total_params/1e6, # in Million 
            "active_parameters": active_params/1e6,
            "base_parameters": base_params/1e6, # Non-MoE params (Embeddings, Attn, Dense Layers)
            "shared_expert_parameters": shared_exp_param/1e6,
            "routed_expert_parameters": routed_exp_param/1e6,
        } 
    else:
        # Standard MoE
        single_expert_size = total_expert_params // num_experts 
        active_params = base_params + (single_expert_size * top_k) 

        return {
            "total_parameters": total_params/1e6,
            "active_parameters": active_params/1e6,
            "base_parameters": base_params/1e6,
            "routed_expert_parameters": total_expert_params/1e6,
        }



In [ ]:
## Model
## Added weight absorption and MoE integration

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        # Upcast to float32 before squaring to prevent FP16 overflow (NaNs)
        norm = x.float().pow(2).mean(dim=-1, keepdim=True) 
        return (x.float() * torch.rsqrt(norm + self.eps)).type_as(x) * self.weight

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0, device="cpu"):
    """ Computing RoPE embeddings """ 
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device) 
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs, device=freqs.device), freqs) 

    return freqs_cis #freqs_cis.to(device)

def precompute_freqs_cis_yarn(
    dim: int, 
    end: int, 
    theta: float = 10000.0, 
    scale: float = 1.0, 
    original_context_length: int = 4096, 
    beta_fast: float = 32.0, 
    beta_slow: float = 1.0, 
    device: str = "cpu"
) -> torch.Tensor:
    """
    YaRN (Yet another RoPE extensioN) scaled frequencies for RoPE.
    
    If scale <= 1.0, this elegantly degrades to the exact original RoPE math.
    Integrates natively with torch.polar for zero-overhead attention scaling.
    """
    # 1. Base frequencies (same as standard RoPE)
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device)[: (dim // 2)].float() / dim))
    
    # 2. YaRN Extrapolation/Interpolation logic
    mscale = 1.0 

    if scale > 1.0:
        print("using YaRN") 
        # YaRN attention scaling multiplier to compensate for entropy changes
        # Applied to both Q and K, resulting in a dot-product scale of mscale^2
        mscale = 0.1 * math.log(scale) + 1.0
        
        # Calculate wavelength boundaries for interpolation
        def find_correction_dim(beta):
            return (dim * math.log(original_context_length / (beta * 2 * math.pi))) / (2 * math.log(theta))
        
        low = max(math.floor(find_correction_dim(beta_fast)), 0)
        high = min(math.ceil(find_correction_dim(beta_slow)), dim // 2 - 1)
        
        # Linear ramp function for frequency blending
        d = torch.arange(dim // 2, dtype=torch.float32, device=device) 
        if low == high:
            ramp_func = torch.ones_like(d, device=device)
        else:
            ramp_func = torch.clamp((d - low) / (high - low), 0.0, 1.0)
            
        # Extrapolate high frequencies (ramp=0), Interpolate low frequencies (ramp=1)
        freqs = freqs * ((1.0 - ramp_func) + ramp_func / scale) 

        # 3. Outer product with position indices
        t = torch.arange(end, device=device)
        freqs = torch.outer(t, freqs).float() 
        
        ## 4. Burn the attention scaling into the complex magnitude
        ## By setting the magnitude to `mscale`, apply_rope inherently scales Q and K
        # mscale_tensor = torch.full_like(freqs, mscale)
        # freqs_cis = torch.polar(mscale_tensor, freqs) 

        # CRITICAL FIX: Magnitude remains 1.0 so apply_rope doesn't alter vector norms
        freqs_cis = torch.polar(torch.ones_like(freqs, device=device), freqs) 
    elif scale == 1.0:
        print("using RoPE") 
        freqs_cis = precompute_freqs_cis(
            dim=dim, 
            end=end, 
            theta=theta,
            device=device
        ) 
    else:
        t = torch.arange(end, device=device)
        freqs = torch.outer(t, freqs).float() 
        
        mscale_tensor = torch.ones_like(freqs, device=device)
        freqs_cis = torch.polar(mscale_tensor, freqs) 
    
    return freqs_cis.to(device), mscale 

def apply_rope_yarn(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor):
    """ Apply RoPE/YaRN scaling """
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, xq_.shape[1], 1, xq_.shape[-1])
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class FeedForwardNet(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__() 
        hidden_dim = args.d_model * args.hidden_dim_multiplier

        self.layers = nn.Sequential(
            nn.Linear(args.d_model, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, args.d_model, bias=False),
        )
        
        self.num_layers = args.num_layers
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # 1. Base initialization: Normal distribution with std=0.02
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
                
            # 2. Residual scaling: Apply to the final projection 
            std = 0.02 / math.sqrt(2.0 * self.num_layers)
            torch.nn.init.normal_(self.layers[2].weight, mean=0.0, std=std) 

    def forward(self, x):
        return self.layers(x) 

class FeedForwardSwiGLU(nn.Module):
    """Standard SwiGLU FeedForward network for non-MoE layers."""
    def __init__(self, args: ModelArgs):
        super().__init__() 
        hidden_dim = args.d_model * args.hidden_dim_multiplier 

        self.w1 = nn.Linear(args.d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(args.d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, args.d_model, bias=False) 

        self.num_layers = args.num_layers
        self.apply(self._init_weights) 
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # 1. Base initialization: Normal distribution with std=0.02
            torch.nn.init.normal_(self.w1.weight, mean=0.0, std=0.02)
            torch.nn.init.normal_(self.w2.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
                
            # 2. Residual scaling: Apply to the final projection 
            # Scale by 1 / sqrt(2 * num_layers) 
            std = 0.02 / math.sqrt(2.0 * self.num_layers)
            torch.nn.init.normal_(self.w3.weight, mean=0.0, std=std) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

## MoE FF Network (Mistral style) 
class MoEFeedForward(nn.Module):
    """Mixture of Experts layer with Z-Loss, Load Balancing, and No zero-padding """
    def __init__(self, moe_args: MoEArgs, emb_dim: int, hidden_dim: int):
        super().__init__()
        self.num_experts = moe_args.num_experts
        self.num_experts_per_tok = moe_args.top_k_experts
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim

        self.z_loss_coef = moe_args.z_loss_coef
        self.moe_loss_coef = moe_args.moe_loss_coef

        self.gate = nn.Linear(emb_dim, self.num_experts, bias=False)

        # Batched weights for fast expert computation: [num_experts, in_features, out_features]
        self.w1 = nn.Parameter(torch.empty(self.num_experts, emb_dim, hidden_dim))
        self.w2 = nn.Parameter(torch.empty(self.num_experts, emb_dim, hidden_dim))
        self.w3 = nn.Parameter(torch.empty(self.num_experts, hidden_dim, emb_dim))

        self._init_weights()

    def _init_weights(self):
        """Manually calculate bounds to match nn.Linear PyTorch initialization exactly.""" 
        nn.init.normal_(self.gate.weight, mean=0.0, std=0.02) 

        bound_w12 = 1 / math.sqrt(self.emb_dim) 
        nn.init.uniform_(self.w1, -bound_w12, bound_w12)
        nn.init.uniform_(self.w2, -bound_w12, bound_w12)

        bound_w3 = 1 / math.sqrt(self.hidden_dim) 
        nn.init.uniform_(self.w3, -bound_w3, bound_w3) 

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch, seq_len, emb_dim = x.shape
        x_flat = x.view(-1, emb_dim) 

        # 1. Routing 
        router_logits = self.gate(x_flat)

        routing_probs = F.softmax(router_logits, dim=-1)
        routing_weights, selected_experts = torch.topk(routing_probs, self.num_experts_per_tok, dim=-1)
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True) 

        aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype) 

        if self.training:
            # Strict Float32 for Loss Calculations (Upcasting prevents logsumexp from overflowing to NaN in fp16/bf16) 
            router_logits_f32 = router_logits.float()
            routing_probs_f32 = routing_probs.float()

            # Z-Loss 
            z_loss = torch.mean(torch.logsumexp(router_logits_f32, dim=-1) ** 2) 

            # OPTIMIZATION: Differentiable, Sync-Free Load Balancing
            # Replaces torch.bincount with static-shaped tensor operations to prevent graph breaks and NCCL timeout stalls during distributed training.
            with torch.no_grad():
                # One-hot encode the selected experts: [num_tokens, top_k, num_experts]
                expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).float()
                
                # Sum across the top_k dimension to get boolean routing: [num_tokens, num_experts]
                expert_mask = expert_mask.sum(dim=1) 
                
                # Fraction of tokens dispatched to each expert (f_i)
                f_i = expert_mask.mean(dim=0) # Shape: [num_experts]

            # Mean routing probability per expert (P_i)
            P_i = routing_probs_f32.mean(dim=0) # Shape: [num_experts]

            # Continuous Load Balancing Loss
            load_balancing_loss = self.num_experts * torch.sum(f_i * P_i)
            
            # Combine losses and cast back to the active model dtype for the backward pass
            aux_loss = (self.moe_loss_coef * load_balancing_loss) + (self.z_loss_coef * z_loss)
            aux_loss = aux_loss.to(x.dtype) 

        ## 3.Y Fast Expert Computation (Best Option for Windows and Linux, with PyTorch EAGER MODE)
        
        flat_expert_indices = selected_experts.view(-1)
        flat_routing_weights = routing_weights.view(-1)
        flat_token_indices = torch.arange(x_flat.size(0), device=x.device).repeat_interleave(self.num_experts_per_tok)

        out_flat = torch.zeros_like(x_flat) 

        for i in range(self.num_experts):
            expert_mask = (flat_expert_indices == i) 
            if not expert_mask.any():
                continue

            # Extract only valid tokens and weights
            valid_token_indices = flat_token_indices[expert_mask]
            curr_tokens = x_flat[valid_token_indices]
            curr_weights = flat_routing_weights[expert_mask].unsqueeze(-1)

            # Dense Matrix Multiplications for specific expert
            h1 = torch.matmul(curr_tokens, self.w1[i])
            h2 = torch.matmul(curr_tokens, self.w2[i])
            hidden = F.silu(h1) * h2
            expert_out = torch.matmul(hidden, self.w3[i])

            # Scale and accumulate
            expert_out = expert_out * curr_weights
            out_flat = out_flat.index_add(0, valid_token_indices, expert_out) 

        return out_flat.view(batch, seq_len, emb_dim), aux_loss 

## MoE-shared expert (DeepSeek)
class DeepSeekSparseMoE(nn.Module):
    """
    DeepSeek-style Mixture of Experts: Shared Experts (always active) + Routed Experts (Top-K selection). 
    Includes FP32 Z-Loss and Sync-Free Load Balancing.
    """
    def __init__(self, moe_args, emb_dim: int, hidden_dim: int):
        super().__init__()
        self.n_routed_experts = moe_args.num_experts
        self.n_shared_experts = moe_args.num_shared_experts
        self.num_experts_per_tok = moe_args.top_k_experts
        
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim 

        self.z_loss_coef = moe_args.z_loss_coef
        self.moe_loss_coef = moe_args.moe_loss_coef

        # ---------------------------------------------------------
        # 1. SHARED EXPERT(S)
        # DeepSeek merges shared experts into one large dense FF 
        # to maximize GPU utilization and avoid routing overhead.
        # ---------------------------------------------------------
        shared_hidden = hidden_dim * self.n_shared_experts
        self.shared_w1 = nn.Linear(emb_dim, shared_hidden, bias=False)
        self.shared_w2 = nn.Linear(emb_dim, shared_hidden, bias=False)
        self.shared_w3 = nn.Linear(shared_hidden, emb_dim, bias=False)

        # ---------------------------------------------------------
        # 2. ROUTED EXPERTS
        # ---------------------------------------------------------
        self.gate = nn.Linear(emb_dim, self.n_routed_experts, bias=False)

        # Batched weights for fast expert computation
        self.w1 = nn.Parameter(torch.empty(self.n_routed_experts, emb_dim, hidden_dim))
        self.w2 = nn.Parameter(torch.empty(self.n_routed_experts, emb_dim, hidden_dim))
        self.w3 = nn.Parameter(torch.empty(self.n_routed_experts, hidden_dim, emb_dim))

        self._init_weights()

    def _init_weights(self):
        # Gate init 
        nn.init.normal_(self.gate.weight, mean=0.0, std=0.02) 

        # Shared Experts init
        nn.init.normal_(self.shared_w1.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.shared_w2.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.shared_w3.weight, mean=0.0, std=0.02)

        # Routed Experts init
        bound_w12 = 1 / math.sqrt(self.emb_dim)
        nn.init.uniform_(self.w1, -bound_w12, bound_w12)
        nn.init.uniform_(self.w2, -bound_w12, bound_w12)

        bound_w3 = 1 / math.sqrt(self.hidden_dim) 
        nn.init.uniform_(self.w3, -bound_w3, bound_w3) 

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch, seq_len, emb_dim = x.shape
        x_flat = x.view(-1, emb_dim) 

        # =========================================================
        # PATH A: Shared Expert Computation (Always Active)
        # =========================================================
        shared_out = self.shared_w3(F.silu(self.shared_w1(x_flat)) * self.shared_w2(x_flat))

        # =========================================================
        # PATH B: Routed Experts Computation
        # =========================================================
        router_logits = self.gate(x_flat)
        routing_probs = F.softmax(router_logits, dim=-1)
        
        routing_weights, selected_experts = torch.topk(routing_probs, self.num_experts_per_tok, dim=-1)
        
        # Normalize top-K weights so they sum to 1
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True) 

        aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)

        if self.training:
            # Sync-Free & FP32 Loss Optimizations
            router_logits_f32 = router_logits.float()
            routing_probs_f32 = routing_probs.float()

            # Z-Loss
            z_loss = torch.mean(torch.logsumexp(router_logits_f32, dim=-1) ** 2)

            # Sync-Free Load Balancing Loss
            with torch.no_grad():
                expert_mask = F.one_hot(selected_experts, num_classes=self.n_routed_experts).float()
                expert_mask = expert_mask.sum(dim=1) 
                f_i = expert_mask.mean(dim=0) 

            P_i = routing_probs_f32.mean(dim=0) 
            load_balancing_loss = self.n_routed_experts * torch.sum(f_i * P_i)
            
            aux_loss = (self.moe_loss_coef * load_balancing_loss) + (self.z_loss_coef * z_loss)
            aux_loss = aux_loss.to(x.dtype)

        # Fast Expert Computation Loop
        flat_expert_indices = selected_experts.view(-1)
        flat_routing_weights = routing_weights.view(-1)
        flat_token_indices = torch.arange(x_flat.size(0), device=x.device).repeat_interleave(self.num_experts_per_tok)

        routed_out_flat = torch.zeros_like(x_flat) 

        for i in range(self.n_routed_experts):
            expert_mask = (flat_expert_indices == i) 
            if not expert_mask.any():
                continue

            valid_token_indices = flat_token_indices[expert_mask]
            curr_tokens = x_flat[valid_token_indices]
            curr_weights = flat_routing_weights[expert_mask].unsqueeze(-1)

            h1 = torch.matmul(curr_tokens, self.w1[i])
            h2 = torch.matmul(curr_tokens, self.w2[i])
            hidden = F.silu(h1) * h2
            expert_out = torch.matmul(hidden, self.w3[i])

            expert_out = expert_out * curr_weights
            routed_out_flat = routed_out_flat.index_add(0, valid_token_indices, expert_out)

        # COMBINE PATHS
        # DeepSeek simply adds the shared representation to the routed representation 
        final_out = shared_out + routed_out_flat

        return final_out.view(batch, seq_len, emb_dim), aux_loss 

## MLA 
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.num_heads 
        self.d_h = args.qk_nope_head_dim 
        self.d_rope = args.qk_rope_head_dim 

        yarn_scale = args.yarn_computed_scale 
        
        # Calculate the mscale factor (matching the YaRN function)
        mscale = (0.1 * math.log(yarn_scale) + 1.0) if yarn_scale > 1.0 else 1.0 
        
        # Base scale * mscale^2 (because it applies to both Q and K)
        self.scale = (1.0 / math.sqrt(self.d_h + self.d_rope)) * (mscale ** 2) 

        # # Scaling factor for the dot product
        # self.scale = 1.0 / math.sqrt(self.d_h + self.d_rope) 

        # KV Compression
        self.kv_down_proj = nn.Linear(args.d_model, args.kv_lora_rank, bias=False)
        self.kv_down_norm = RMSNorm(args.kv_lora_rank)
        self.k_up_proj = nn.Linear(args.kv_lora_rank, args.num_heads * self.d_h, bias=False)
        self.v_up_proj = nn.Linear(args.kv_lora_rank, args.num_heads * args.v_head_dim, bias=False)
        self.k_rope_proj = nn.Linear(args.d_model, self.d_rope, bias=False)

        # Query Compression
        self.q_down_proj = nn.Linear(args.d_model, args.q_lora_rank, bias=False)
        self.q_down_norm = RMSNorm(args.q_lora_rank)
        self.q_up_proj = nn.Linear(args.q_lora_rank, args.num_heads * self.d_h, bias=False)
        self.q_rope_proj = nn.Linear(args.q_lora_rank, args.num_heads * self.d_rope, bias=False)

        self.out_proj = nn.Linear(args.num_heads * args.v_head_dim, args.d_model, bias=False) 

        # Injected runtime attributes for Absorbed Inference Graph
        self.fused_q_proj = None
        self.fused_v_out = None 
    
    def absorb_weights(self):
        """
        Fuses the KV up-projection weights into the Query and Output projections.
        Run this EXACTLY ONCE before starting inference.
        """
        # 1. Absorb W_UK into W_UQ
        # W_uq: (n_heads * d_h, q_lora_rank) -> (n_heads, d_h, q_lora_rank)
        W_uq = self.q_up_proj.weight.view(self.n_heads, self.d_h, -1)
        # W_uk: (n_heads * d_h, kv_lora_rank) -> (n_heads, d_h, kv_lora_rank)
        W_uk = self.k_up_proj.weight.view(self.n_heads, self.d_h, -1)
        
        # Fused Q-K projection: (n_heads, q_lora_rank, kv_lora_rank) (TODO matrix optimization)
        self.fused_q_proj = torch.bmm(W_uq.transpose(1, 2), W_uk) 
        
        # 2. Absorb W_UV into W_OUT
        # W_uv: (n_heads * v_head_dim, kv_lora_rank) -> (n_heads, v_head_dim, kv_lora_rank)
        # Assuming args.v_head_dim is available, otherwise replace with self.d_h if they are identical
        v_dim = self.out_proj.weight.shape[1] // self.n_heads
        W_uv = self.v_up_proj.weight.view(self.n_heads, v_dim, -1)
        # W_o: (d_model, n_heads * v_head_dim) -> (d_model, n_heads, v_head_dim)
        W_o = self.out_proj.weight.view(-1, self.n_heads, v_dim)
        
        # Fused O-V projection: (n_heads, d_model, kv_lora_rank)  (TODO matrix optimization) 
        self.fused_v_out = torch.einsum('d h v, h v k -> h d k', W_o, W_uv) 
    
    @torch.no_grad()
    def forward_inference(self, x: torch.Tensor, freqs_cis: torch.Tensor, kv_cache: Tuple[torch.Tensor, torch.Tensor] = None):
        """
        Optimized path utilizing compressed KV structures and weight absorption.
        """
        B, L, _ = x.shape
        
        # 1. Compute latent states and absolute RoPE projections for the new token(s)
        c_kv = self.kv_down_norm(self.kv_down_proj(x)) 
        k_rope = self.k_rope_proj(x).view(B, L, 1, self.d_rope)
        
        c_q = self.q_down_norm(self.q_down_proj(x))
        q_rope = self.q_rope_proj(c_q).view(B, L, self.n_heads, self.d_rope)
        
        # 2. Apply RoPE BEFORE caching so historical keys maintain their absolute position rotations
        q_rope, k_rope = apply_rope_yarn(q_rope, k_rope, freqs_cis)
        
        # 3. Cache Update
        if kv_cache is not None:
            c_kv_cache, k_rope_cache = kv_cache
            c_kv = torch.cat([c_kv_cache, c_kv], dim=1)
            k_rope = torch.cat([k_rope_cache, k_rope], dim=1)
        
        new_cache = (c_kv, k_rope)
        total_len = c_kv.size(1)
        
        # 4. Compute Attention Scores (Separated into NOPE and RoPE components)
        
        # NOPE Part: (B, H, L, kv_rank) x (B, 1, kv_rank, total_len) -> (B, H, L, total_len)
        q_nope_fused = torch.einsum('b l q, h q k -> b h l k', c_q, self.fused_q_proj)
        scores_nope = torch.matmul(q_nope_fused, c_kv.unsqueeze(1).transpose(-2, -1))
        
        # RoPE Part: (B, H, L, d_rope) x (B, 1, d_rope, total_len) -> (B, H, L, total_len)
        q_rope = q_rope.transpose(1, 2)
        k_rope = k_rope.transpose(1, 2)
        scores_rope = torch.matmul(q_rope, k_rope.transpose(-2, -1))
        
        # Total scaled attention matrix
        scores = (scores_nope + scores_rope) * self.scale
        
        # 5. Dynamic Causal Masking (Critical for the L > 1 Prefill Step)
        if L > 1:
            causal_mask = torch.triu(
                torch.full((L, total_len), float('-inf'), device=scores.device), 
                diagonal=total_len - L + 1
            )
            scores = scores + causal_mask
            
        attn_weights = F.softmax(scores.to(torch.float32), dim=-1).type_as(x)
        
        # 6. Value Gathering and Fused Linear Back-projection
        attn_cv = torch.matmul(attn_weights, c_kv.unsqueeze(1)) # (B, H, L, kv_rank)
        out = torch.einsum('b h l k, h d k -> b l d', attn_cv, self.fused_v_out)
        
        return out, new_cache
    
    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor):
        B, L, _ = x.shape 
        
        # 1. KV Path
        c_kv = self.kv_down_norm(self.kv_down_proj(x))
        k_nope = self.k_up_proj(c_kv).view(B, L, self.n_heads, self.d_h)
        v = self.v_up_proj(c_kv).view(B, L, self.n_heads, self.d_h)
        k_rope = self.k_rope_proj(x).view(B, L, 1, self.d_rope)

        # 2. Query Path
        c_q = self.q_down_norm(self.q_down_proj(x))
        q_nope = self.q_up_proj(c_q).view(B, L, self.n_heads, self.d_h)
        q_rope = self.q_rope_proj(c_q).view(B, L, self.n_heads, self.d_rope) 

        # 3. Apply RoPE
        q_rope, k_rope = apply_rope_yarn(q_rope, k_rope, freqs_cis)
        q_rope = q_rope.expand(-1, -1, self.n_heads, -1)
        k_rope = k_rope.expand(-1, -1, self.n_heads, -1)

        # 4. Concatenate and Transpose for Matrix Multiplication
        # Shape becomes: (Batch, Heads, SeqLen, HeadDim + RopeDim)
        q = torch.cat([q_nope, q_rope], dim=-1).transpose(1, 2)
        k = torch.cat([k_nope, k_rope], dim=-1).transpose(1, 2)
        v = v.transpose(1, 2) 


        ## 5. MANUAL SCALED DOT-PRODUCT ATTENTION
        
        # Q * K^T. We transpose only the last two dimensions of K.
        # q: (B, H, L, D) | k.transpose: (B, H, D, L) -> scores: (B, H, L, L)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        # Create a causal mask dynamically matching the current sequence length
        causal_mask = torch.triu(
            torch.full((L, L), float('-inf'), device=q.device), 
            diagonal=1
        )
        
        # Upcast scores to float32 before adding mask and applying softmax to prevent NaNs
        scores = scores.to(torch.float32) + causal_mask
        
        # Apply Softmax to get probabilities (B, H, L, L)
        attn_weights = F.softmax(scores, dim=-1)
        
        # Cast back to the original precision (e.g., float16 or bfloat16) before multiplying by V
        attn_weights = attn_weights.type_as(q)
        
        # Multiply by V: (B, H, L, L) x (B, H, L, D) -> (B, H, L, D)
        attn_out = torch.matmul(attn_weights, v)
        
        # Transpose back and project to dense model dimension
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.out_proj(attn_out) 

## Decoder block 
class TransformerDecoderLayer(nn.Module):
    def __init__(self, args: ModelArgs, layer_idx: int):
        super().__init__()
        self.attn = MultiHeadLatentAttention(args) 

        # FeedForward Init 
        if args.moe_args and layer_idx >= args.moe_args.first_moe_layer:
            if args.moe_args.use_shared_expert:
                self.ffn = DeepSeekSparseMoE(args.moe_args, args.d_model, args.d_model * args.moe_args.hidden_dim_multiplier_perexp) 
            else:
                self.ffn = MoEFeedForward(args.moe_args, args.d_model, args.d_model * args.moe_args.hidden_dim_multiplier_perexp) 
            self.is_moe = True
        else:
            self.ffn = FeedForwardNet(args) # FeedForwardSwiGLU(args) 
            self.is_moe = False

        self.norm1 = RMSNorm(args.d_model)
        self.norm2 = RMSNorm(args.d_model)
    
    def forward_inference(self, x: torch.Tensor, freqs_cis: torch.Tensor, kv_cache=None):
        """Inference forward pass bypassing the standard forward graph."""
        # 1. Attention with Compressed Cache & Absorbed Weights
        attn_out, new_kv_cache = self.attn.forward_inference(
            self.norm1(x), freqs_cis, kv_cache=kv_cache
        )
        x = x + attn_out
        
        # 2. FFN Block (MoE or Dense)
        if self.is_moe:
            # During generation inference, auxiliary losses are typically not optimized
            ffn_out, _ = self.ffn(self.norm2(x))
            x = x + ffn_out
        else:
            x = x + self.ffn(self.norm2(x))
            
        return x, new_kv_cache
    
    def forward(self, x, freqs_cis):
        x = x + self.attn(self.norm1(x), freqs_cis) 

        # FFN Block with Auxiliary Loss routing
        aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)  
        if self.is_moe:
            ffn_out, aux_loss = self.ffn(self.norm2(x))
            x = x + ffn_out
        else:
            x = x + self.ffn(self.norm2(x))
        
        return x, aux_loss 

## Main Model 
class MimiKoKo_M1(nn.Module):
    """The main language model combining token embedding, blocks, and language modeling head.""" 
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.token_embeddings = nn.Embedding(args.vocab_size, args.d_model)

        self.layers = nn.ModuleList([TransformerDecoderLayer(args, layer_idx=i) for i in range(args.num_layers)])
        self.final_norm = RMSNorm(args.d_model)
        self.lm_head = nn.Linear(args.d_model, args.vocab_size, bias=False) 

        if args.moe_args:
            print("Using MoE...")
        else:
            print("Using FF...") 
        
        # Track routing structural variables internally
        self._current_aux_loss = 0.0

        # Apply standard initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # Proper GPT-style weight initialization to prevent activation explosion
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

        # Scale residual projections to manage deep network variance
        for pn, p in self.named_parameters():
            if pn.endswith('out_proj.weight'): # or pn.endswith('ffn.2.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.args.num_layers))
    
    def forward(self, tokens: torch.Tensor, freqs_cis: torch.Tensor):
        x = self.token_embeddings(tokens) 
        total_aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype) if self.args.moe_args else 0.0 

        for layer in self.layers:
            x, aux_loss = layer(x, freqs_cis) 
            if self.args.moe_args:
                total_aux_loss += aux_loss.to(total_aux_loss.dtype) 
        
        self._current_aux_loss = total_aux_loss

        return self.lm_head(self.final_norm(x)) 
    
    def get_aux_loss(self) -> torch.Tensor:
        """Exposes tracking data seamlessly to the inner stepping execution loop."""
        return self._current_aux_loss 
    
    def absorb_weights(self):
        """Triggers weight fusion on every MLA block. Call this once before generation starts."""
        for layer in self.layers:
            layer.attn.absorb_weights() 



In [ ]:
## Train

import time
import yaml
from torch.utils.data import DataLoader
from dataclasses import asdict
import torch.nn.functional as F
from torch.optim import AdamW


torch.cuda.empty_cache()
torch.manual_seed(123)

paths = ProjectPaths()
model_args = ModelArgs()
train_args = TrainingArgs()


print(f"Training Device: {model_args.device}")

os.makedirs(paths.model_checkpoints_dir, exist_ok=True)

# Instantiate datasets and generators
dataset = TextTokenDataset(paths.data_corpus, model_args.org_context_length, paths.tokenizer_dir)
dataloader = DataLoader(dataset, batch_size=train_args.batch_size, shuffle=True, drop_last=True)

## Setting MoE args 
model_args.moe_args = MoEArgs(
    num_experts=4, 
    top_k_experts=1, 
    hidden_dim_multiplier_perexp=1, 
    num_shared_experts=1,
    use_shared_expert=True 
) 

model_args.vocab_size = dataset.safe_vocab_size 
# print(model_args) 

# Precompute positional tensors & attention structures 
freqs_cis, mscale = precompute_freqs_cis_yarn(
    dim = model_args.qk_rope_head_dim,   # In MLA, only d_rope receives rotation
    end = model_args.org_context_length,          # e.g., 4096 (or 32768 for extended)
    theta = model_args.rotary_freq_base,       # usually 10000.0 or 1000000.0
    scale = model_args.scale_factor,       # 1.0 for base, e.g., 4.0 for 4x context
    original_context_length = model_args.org_context_length, 
    device= model_args.device
) 
model_args.yarn_computed_scale = mscale 

model = MimiKoKo_M1(model_args).to(model_args.device) 

## Param count 
tot_param = sum(p.numel() for p in model.parameters()) / 1e6 
print(f"Total parameters: {tot_param:.2f}M") 
if model_args.moe_args:
    moe_param_count = calculate_active_params_MoE(model)
    active_param = moe_param_count['active_parameters'] 
    shared_param = moe_param_count['shared_expert_parameters'] if "shared_expert_parameters" in moe_param_count else 0 
    print(f"Active: {active_param}M | Shared: {shared_param}M") 
    # print(moe_param_count) 

# Optimizer with Cosine scheduler 
optimizer = AdamW(model.parameters(), lr=train_args.lr_rate, betas=(0.9, 0.990)) 

updates_per_epoch = math.ceil(
    len(dataloader) / train_args.grad_accum_steps 
) 
total_updates = (updates_per_epoch) * train_args.epochs 
total_steps = total_updates//2 

scheduler = get_cosine_schedule_with_warmup(optimizer, train_args.warmup_steps, total_updates) 

steps_per_epoch = len(dataloader) 
print(f"{steps_per_epoch} | {total_updates}") 

model.train() 
st_time = time.perf_counter()
net_step_time = 0 
step_time = 0 
global_step = 0 

print("Beginning Training Iterations...\n")

for ep in range(train_args.epochs):
    for batch_idx, (x, y) in enumerate(dataloader):
        t0 = time.perf_counter() 
        x, y = x.to(model_args.device), y.to(model_args.device)

        # Autocast automatically manages safe dtype casting
        with torch.autocast(device_type=model_args.device.type, dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, enabled=model_args.device.type == "cuda"):
            logits = model(x, freqs_cis)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)) 

            if model_args.moe_args:
                loss = loss + model.get_aux_loss() 
            loss = loss / train_args.grad_accum_steps
        
        # scaler.scale(loss).backward()
        loss.backward()

        if (batch_idx + 1) % train_args.grad_accum_steps == 0:
            nn.utils.clip_grad_norm_(model.parameters(), train_args.grad_clip)
            
            optimizer.step() 
            scheduler.step() 
            optimizer.zero_grad(set_to_none=True) 

        global_step += 1 
        
        step_time = time.perf_counter() - t0
        net_step_time += step_time

        if batch_idx % 10 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch: {ep} (BID:{batch_idx}) | Step: {global_step}/{total_steps} | Step_time: {step_time:.6f} s | Loss: {loss.item() * train_args.grad_accum_steps:.4f} | LR: {current_lr:.6f}") 

        # Save Checkpoints
        if global_step > 0 and global_step % train_args.save_every_steps == 0:
            checkpoint_path = os.path.join(paths.model_checkpoints_dir, f"mla_step_{global_step}.pt")
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, checkpoint_path)
            print(f"Saved checkpoint model artifact at: {checkpoint_path}") 

        if global_step >= total_steps:
            break 

        if train_args.tot_steps > 1:
            if batch_idx >= train_args.tot_steps:
                print(f"{train_args.tot_steps} steps reached") 
                break 
    
    preds = torch.argmax(logits, dim=-1)
    correct_mask = (preds == y)
    total_correct = correct_mask.sum().item()
    total_tokens = y.numel()
    print(f"Epoch:{ep} | Accuracy: {(total_correct/total_tokens) * 100}")

tot_time = time.perf_counter() - st_time
last_loss = loss.item()
print(f"\nTraining completed successfully in: {tot_time:.5f}s") 

# Final Model Storage Integration 
model_final_path = os.path.join(paths.model_checkpoints_dir, "MLA_V5_moe_v1.pt") 
config_path = os.path.join(paths.model_checkpoints_dir, 'training_config_MLA_V5_moe_v1.yaml') 

model_args.device = str(torch.device("cuda" if torch.cuda.is_available() else "cpu")) 

training_params = {
    "hyperparameters": asdict(model_args),
    "training_args": asdict(train_args),
    "training_stats": {
        'total_steps': total_steps,
        'step_time_s': float(f"{step_time:.6f}"),
        'loss': float(f"{last_loss:.6f}"),
        'training_time_s': float(f"{tot_time:.4f}") 
    }, 
    "total_param": f"{tot_param:.2f}M",
    "active_param": f"{active_param:.2f}M" if model_args.moe_args else None 
} 

try:
    with open(config_path, 'w') as file:
        yaml.dump(training_params, file)
except Exception as exc:
    print(f"Error saving config file! \n{exc}")

torch.save(model.state_dict(), model_final_path)
print(f"Model artifacts saved at: {model_final_path}")

del model


### Inference  


In [ ]:
## Init

import os
import time
import yaml
import torch
from transformers import AutoTokenizer
import torch.nn as nn
import torch.nn.functional as F

torch.cuda.empty_cache() 

## Min-P and Temperature Scaling Sampling Logic 
def sample_min_p(logits: torch.Tensor, temperature: float = 1.0, min_p: float = 0.05) -> torch.Tensor:
    """
    Applies temperature scaling and filters out tokens with probabilities 
    less than min_p * max_prob.
    """
    # 1. Apply Temperature Scaling
    if temperature > 0.0:
        logits = logits / temperature
    else:
        # Greedy fallback if temperature is explicitly zero
        return torch.argmax(logits, dim=-1, keepdim=True)

    probs = F.softmax(logits, dim=-1)
    max_probs, _ = torch.max(probs, dim=-1, keepdim=True)
    
    # 2. Define the Min-P cutoff threshold dynamically per batch elements
    min_p_threshold = max_probs * min_p
    
    # Filter logits whose raw target probabilities fall under the adaptive ceiling threshold
    masked_logits = torch.where(probs < min_p_threshold, torch.full_like(logits, float("-inf")), logits)
    
    # 3. Sample from modified filtered distribution
    filtered_probs = F.softmax(masked_logits, dim=-1)
    next_token = torch.multinomial(filtered_probs, num_samples=1) 

    _max, _ = filtered_probs.max(dim=-1, keepdim=True)
    token_conf = _max.item() 
    if token_conf <= 0.90:
        print("low  conf.") 

    return next_token 

## Generation Engine Setup 
@torch.no_grad()
def generate(model: MimiKoKo_M1, prompt_text: str, tokenizer_dir: str, max_new_tokens: int = 50, safe_context_limit:int = 1024, temperature: float = 0.7, min_p: float = 0.05, device: str = "cpu"):
    # 1. Fuse weights exactly once before starting the generation loop
    model.absorb_weights() 
    model.eval()
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
    except Exception:
        from transformers import GPT2TokenizerFast
        tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

    # Encode prompt string to long tensor
    input_ids = torch.tensor(tokenizer.encode(prompt_text), dtype=torch.long, device=device).unsqueeze(0) 
    
    prompt_len = input_ids.size(1)
    max_seq_len = prompt_len + max_new_tokens
    
    # Enforce protection boundary for context allocation limits
    max_supported_len = model.args.org_context_length * model.args.scale_factor
    if max_seq_len > max_supported_len:
        print(f"Warning: Requested length {max_seq_len} exceeds YaRN limit {max_supported_len}. Clipping tokens.")
        max_new_tokens = max(0, max_supported_len - prompt_len) 
        max_seq_len = prompt_len + max_new_tokens  
    
    ## Limiting gen tokens to a safe limit 
    if max_seq_len > (safe_context_limit + prompt_len):
        raise ValueError(f"Safe generation limit Exceded. Limit: {safe_context_limit}") 
        # max_seq_len = prompt_len + safe_context_limit 
    
    print(f"\n--- Generating text for input prompt: '{prompt_text}' ---")
    
    # 2. Precompute RoPE/YaRN for the maximum possible length of this sequence 
    
    # freqs_cis_all = precompute_freqs_cis(
    #     dim=model.args.qk_rope_head_dim, 
    #     end=max_seq_len, 
    #     theta=model.args.rotary_freq_base, 
    #     device=device
    # ) 
    freqs_cis_all, _ = precompute_freqs_cis_yarn(
        dim=model.args.qk_rope_head_dim, 
        end=max_seq_len, 
        theta=model.args.rotary_freq_base, 
        scale=model.args.scale_factor,
        original_context_length=model.args.org_context_length,
        device=device
    ) 
    
    # 3. Initialize empty KV cache state for each layer
    kv_caches = [None] * len(model.layers)
    
    for t in range(max_new_tokens):
        current_len = input_ids.size(1)
        
        # 4. Context Slicing & Incremental Forward Strategy
        if t == 0:
            # Prefill Phase: Process the entire prompt at once
            tokens_to_process = input_ids
            freqs_cis = freqs_cis_all[:prompt_len]
        else:
            # Decode Phase: Process only the single newly generated token
            tokens_to_process = input_ids[:, -1:] 
            freqs_cis = freqs_cis_all[current_len - 1 : current_len]
        
        # 5. Execute the optimized Inference Path
        x = model.token_embeddings(tokens_to_process)
        
        for i, layer in enumerate(model.layers):
            # Pass and update the rolling cache for this specific layer
            x, kv_caches[i] = layer.forward_inference(x, freqs_cis, kv_cache=kv_caches[i])
            
        # 6. Gather the last step's predicting logits: (Batch=1, SeqLen, Vocab) -> (Vocab)
        # Slicing x[:, -1, :] ensures we only pull the final step even during the prefill phase
        logits = model.lm_head(model.final_norm(x[:, -1, :])) 
        next_token_logits = logits[0]
        
        # Scale and Sample
        next_token = sample_min_p(next_token_logits, temperature=temperature, min_p=min_p)
        
        # Concatenate to rolling sequence tracking window
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
        
        # Terminate immediately if end-of-string token generated
        if next_token.item() == tokenizer.eos_token_id:
            break

    # Decode complete output tracking window safely back to clean text string
    output_text = tokenizer.decode(input_ids[0].tolist(), skip_special_tokens=True) 

    return output_text


paths = ProjectPaths() 

tokenizer = AutoTokenizer.from_pretrained(os.path.abspath(paths.tokenizer_dir)) 
config_path = os.path.join(paths.model_checkpoints_dir, 'training_config_MLA_moe_v2.yaml') 
_model_path = os.path.join(paths.model_checkpoints_dir, "MLA_final_moe_v2.pt") 

with open(config_path, 'r') as file:
    configs = yaml.safe_load(file) 

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Configuration file not found at {config_path}. Did you run train.py first?")

hyperparams = configs["hyperparameters"] 
if "moe_args" in hyperparams and hyperparams["moe_args"] is not None:
    hyperparams["moe_args"] = MoEArgs(**hyperparams["moe_args"])

model_args = ModelArgs(**hyperparams) 
# print(model_args) 
# model_args.scale_factor = model_args.scale_factor + 1.0   # Max the scaling can be increased by factor of 1 during inference 

model = MimiKoKo_M1(model_args).to(model_args.device) 

if os.path.exists(_model_path):
    # print(f"\nLoading weights matrix from saved checkpoint: {_model_path}")
    model.load_state_dict(torch.load(_model_path, map_location=model_args.device))
else:
    print("[! WARN]: No pre-existing checkpoint path detected. Running evaluation on random initialization matrix weights.")

## For checkpoint loading 
# final_model_path = os.path.join(paths.model_checkpoints_dir, "mla_final_model_v2.pt") 
# model.load_state_dict(torch.load(final_model_path, map_location=model_args.device, weights_only=False)['model_state_dict']) 

safe_context_len = int(0.6875 * (model_args.org_context_length * model_args.scale_factor)) 


Using MoE...


In [ ]:
# Execution Arguments
prompt = "SAMPLE 6: Docker Multi-Stage Build" 

gen_st_time = time.perf_counter() 
generated_text = generate(
    model=model,
    prompt_text=prompt.strip(),
    tokenizer_dir=paths.tokenizer_dir,
    max_new_tokens=1536, 
    safe_context_limit=safe_context_len,
    temperature=0.5, 
    min_p=0.1, 
    device=model_args.device
) 
tot_gen_time = time.perf_counter() - gen_st_time 

print(f"Response >>> \n{generated_text}")

resp_token_ids = tokenizer.encode(generated_text)
print(f"Response Token len: {len(resp_token_ids)}")
print(f"{int(len(resp_token_ids)/tot_gen_time)} tokens/sec") 
